# Notebook 04: Adversarial Defenses

## Why This Matters
Seven of nine adversarial defense papers accepted at ICLR 2018 were broken within months of publication. The pattern wasn't random failure — it was a systematic blind spot: defenses were evaluated against the same weak attacks that motivated them, not against adaptive adversaries who know the defense exists. Understanding why defenses fail is as important as understanding why attacks work.

This notebook catalogs the defense landscape: what was proposed, what was broken and why, what actually provides guarantees, and what the current state of practice looks like. The goal is to leave you with a principled framework for evaluating any claimed defense — rather than a list of techniques to memorize.

## Structure
1. The defense taxonomy: detection vs. robustness vs. certified
2. Input preprocessing defenses — and why they fail
3. Detection-based defenses — and the adaptive attacker
4. Adversarial training variants (FGSM-AT, PGD-AT, TRADES)
5. Certified defenses: randomized smoothing
6. The accuracy-robustness tradeoff — is it fundamental?
7. Practical defense checklist

## What You'll Learn
- Why obfuscated gradients are security theater and how to detect them
- Why TRADES achieves better robustness-accuracy tradeoff than vanilla PGD-AT
- How randomized smoothing provides *provable* robustness certificates
- Why the accuracy-robustness tradeoff is provably fundamental, not just an artifact of training
- The three questions to ask of any claimed defense

## Background

### The 2018 breaking spree

After FGSM and PGD demonstrated that neural networks were systematically vulnerable to adversarial perturbations, the research community produced a wave of proposed defenses in 2016–2018. These defenses were varied and creative: input transformations, detection networks, stochastic components, feature squeezing, input reconstruction via autoencoders, and more.

Athalye, Carlini & Wagner (NeurIPS 2018, best paper) systematically evaluated nine of these defenses — seven accepted at ICLR 2018 — and broke all but two. Their paper, *"Obfuscated Gradients Give a False Sense of Security"*, identified the common failure mode: most defenses worked by making the gradient landscape look uninformative to the attacker, not by actually reducing the set of adversarial examples.

They characterized three types of gradient obfuscation:
1. **Shattered gradients**: non-differentiable operations (quantization, rounding) that produce NaN or zero gradients
2. **Stochastic gradients**: random operations (random resizing, random noise addition) that produce different gradients on each forward pass
3. **Exploding/vanishing gradients**: operations that numerically saturate the gradient, making it useless

The fix for all three: adaptive attacks. If you know the defense exists, you can design an attack that works around it — differentiating through stochastic operations via the reparameterization trick, using Backward Pass Differentiable Approximation (BPDA) to approximate non-differentiable operations, or using finite-difference gradient estimates.

The lesson that was hard for the community to absorb: **security evaluation requires thinking like an adversary who knows your defense**. Evaluating a defense against the same attacker that motivated it is circular.

### The arms race structure

Adversarial ML has an arms race structure that's different from classical security. In classical security, a vulnerability is typically patched and fixed. In adversarial ML, the attack and defense are locked in an optimization game — improving the defense raises the bar for the attacker, but the attacker can adapt. No defense has been found that provably eliminates all adversarial examples for non-trivial models.

This led the community in two directions:
1. **Empirical robustness**: adversarial training variants (PGD-AT, TRADES, MART) that don't provide guarantees but raise the practical bar significantly
2. **Certified robustness**: randomized smoothing and interval bound propagation that provide *provable* certificates at the cost of clean accuracy

### TRADES: a better adversarial training objective

PGD-AT minimizes the loss on adversarial examples directly. Zhang et al. (2019) pointed out that this conflates two objectives: clean accuracy and robustness. Their TRADES (TRadeoff-inspired Adversarial DEfense via Surrogate-loss minimization) formulation separates them:

$$\min_\theta \mathbb{E} \left[ \mathcal{L}(f_\theta(x), y) + \frac{1}{\lambda} \max_{\delta \in \mathcal{B}} \text{KL}(f_\theta(x) \| f_\theta(x + \delta)) \right]$$

The first term is clean accuracy. The second term is a regularizer that penalizes the model for changing its output distribution when inputs are perturbed. The hyperparameter $\lambda$ controls the tradeoff. This produces better robustness at the same clean accuracy compared to PGD-AT, and it provides cleaner theoretical analysis of the tradeoff.

### Randomized smoothing: the first scalable certified defense

Cohen, Rosenfeld & Kolter (ICML 2019) introduced the first certified defense that scales to ImageNet: randomized smoothing. The idea: instead of classifying $x$ directly, classify the average prediction of $f$ over Gaussian noise added to $x$:

$$g(x) = \arg\max_c \Pr_{\delta \sim \mathcal{N}(0, \sigma^2 I)}[f(x + \delta) = c]$$

Cohen et al. showed that if $g(x)$ predicts class $c$ with probability $p_A$ and the runner-up class with probability $p_B$, then $g$ is certifiably robust within $L_2$ radius:

$$R = \frac{\sigma}{2} (\Phi^{-1}(p_A) - \Phi^{-1}(p_B))$$

where $\Phi^{-1}$ is the inverse Gaussian CDF. No attack can change the prediction of $g$ on $x$ if the perturbation is smaller than $R$. This is a *proof*, not an empirical observation.

The cost: predictions are stochastic (you need Monte Carlo sampling to estimate $g$), and clean accuracy drops because you're averaging over noisy inputs during both training and inference.

In [ ]:
%pip install -q torch torchvision matplotlib numpy scipy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as scipy_norm

torch.manual_seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset,  batch_size=256, shuffle=False)
print("Data loaded.")

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


def evaluate(model, loader, device, n_batches=None):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if n_batches and i >= n_batches: break
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total


def pgd_attack(model, x, y, epsilon, alpha, n_steps):
    model.eval()
    delta = torch.zeros_like(x).uniform_(-epsilon, epsilon).requires_grad_(True)
    for _ in range(n_steps):
        loss = F.cross_entropy(model(x + delta), y)
        loss.backward()
        with torch.no_grad():
            delta_new = (delta + alpha * delta.grad.sign()).clamp(-epsilon, epsilon)
            delta_new = torch.clamp(x + delta_new, x.min(), x.max()) - x
            delta = delta_new.requires_grad_(True)
    return (x + delta).detach()


# Train baseline clean model
model_clean = MnistCNN().to(device)
opt = torch.optim.Adam(model_clean.parameters(), lr=1e-3)
print("Training clean model...")
for epoch in range(5):
    model_clean.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); F.cross_entropy(model_clean(x), y).backward(); opt.step()
print(f"Clean model accuracy: {evaluate(model_clean, test_loader, device):.4f}")

## 1. Why Preprocessing Defenses Fail

The most intuitive defense idea: clean up the adversarial perturbation before it reaches the model. Compress the image, smooth it, or project it onto a lower-dimensional manifold. The perturbation is small and high-frequency — preprocessing might remove it.

The failure: if the preprocessing is differentiable (even approximately), the attacker can compute gradients *through* it and craft perturbations that survive preprocessing. This is called an **adaptive attack** — an attack designed with full knowledge of the defense.

The key insight from Athalye et al.: the right question isn't "does this defense defeat FGSM?" — it's "does this defense defeat an attacker who knows the defense exists and designs their attack accordingly?"

In [ ]:
def gaussian_smooth(x: torch.Tensor, sigma: float = 0.5) -> torch.Tensor:
    """Differentiable Gaussian smoothing via separable convolution."""
    # Build Gaussian kernel
    kernel_size = 5
    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
    kernel_1d = torch.exp(-0.5 * (coords / sigma) ** 2)
    kernel_1d = kernel_1d / kernel_1d.sum()
    kernel_2d = kernel_1d.outer(kernel_1d).unsqueeze(0).unsqueeze(0)
    kernel_2d = kernel_2d.expand(x.size(1), 1, kernel_size, kernel_size)
    return F.conv2d(x, kernel_2d, padding=kernel_size // 2, groups=x.size(1))


def pgd_through_defense(model, defense_fn, x, y, epsilon, alpha, n_steps):
    """Adaptive PGD: gradients flow through the defense function."""
    model.eval()
    delta = torch.zeros_like(x).uniform_(-epsilon, epsilon).requires_grad_(True)
    for _ in range(n_steps):
        # Forward: defense is applied BEFORE the model
        loss = F.cross_entropy(model(defense_fn(x + delta)), y)
        loss.backward()
        with torch.no_grad():
            delta_new = (delta + alpha * delta.grad.sign()).clamp(-epsilon, epsilon)
            delta_new = torch.clamp(x + delta_new, x.min(), x.max()) - x
            delta = delta_new.requires_grad_(True)
    return (x + delta).detach()


# Evaluate: naive attack vs adaptive attack against smoothing defense
x_b, y_b = next(iter(test_loader))
x_b, y_b = x_b[:128].to(device), y_b[:128].to(device)
epsilon = 0.3

x_pgd_naive    = pgd_attack(model_clean, x_b, y_b, epsilon, 0.01, 40)
x_pgd_adaptive = pgd_through_defense(model_clean, gaussian_smooth, x_b, y_b, epsilon, 0.01, 40)

with torch.no_grad():
    # Evaluate with defense active
    acc_clean      = (model_clean(gaussian_smooth(x_b)).argmax(1) == y_b).float().mean().item()
    acc_naive_def  = (model_clean(gaussian_smooth(x_pgd_naive)).argmax(1) == y_b).float().mean().item()
    acc_adapt_def  = (model_clean(gaussian_smooth(x_pgd_adaptive)).argmax(1) == y_b).float().mean().item()

print("Gaussian smoothing defense evaluation:")
print(f"  Clean accuracy (with smoothing):          {acc_clean:.4f}")
print(f"  Naive PGD accuracy (with smoothing):      {acc_naive_def:.4f}  ← defense appears to work")
print(f"  Adaptive PGD accuracy (with smoothing):   {acc_adapt_def:.4f}  ← defense is broken")
print()
print("The naive attack doesn't account for smoothing — perturbations are partially removed.")
print("The adaptive attack computes gradients through smoothing — perturbations survive.")
print()
print("Lesson: any differentiable defense can be broken by an attacker who knows it exists.")

## 2. The Three Signs of Gradient Masking

Athalye et al. identified three diagnostic signs that a defense is relying on gradient masking rather than genuine robustness. Any defense showing these signs should be treated with suspicion until adaptive attacks are evaluated.

1. **More PGD steps → less attack success**: a genuine adversarial defense should become harder to evade with more attack steps (or plateau). If more steps actually *help* the defense, the gradient is being used in the wrong direction.

2. **Transferability: black-box attack outperforms white-box**: if adversarial examples crafted against a surrogate model (no access to the defense) fool the defended model more than the white-box attack does, the defense is masking gradients rather than reducing the adversarial region.

3. **Random noise nearly as effective as FGSM**: if random noise achieves similar attack success to gradient-based attacks, the gradient signal is not useful — it's being masked.

In [ ]:
def run_gradient_masking_diagnostics(model, defense_fn, x, y, epsilon, device):
    """
    Runs the three Athalye et al. gradient masking diagnostic tests.
    A legitimate defense should pass all three.
    """
    model.eval()
    results = {}

    print("Gradient masking diagnostics:")
    print("=" * 55)

    # Test 1: More PGD steps → more attack success (not less)
    print("\n[Test 1] Attack success vs number of PGD steps")
    print("  Expect: monotonically increasing (or plateau)")
    print("  Red flag: decreasing attack success with more steps")
    prev_success = 0
    flag_1 = False
    for n_steps in [1, 5, 10, 20, 40]:
        alpha = 2.5 * epsilon / n_steps
        x_adv = pgd_through_defense(model, defense_fn, x, y, epsilon, alpha, n_steps)
        with torch.no_grad():
            success = (model(defense_fn(x_adv)).argmax(1) != y).float().mean().item()
        status = "↓ FLAG" if success < prev_success - 0.02 else ""
        if status:
            flag_1 = True
        print(f"  steps={n_steps:>3}: attack success = {success:.4f} {status}")
        prev_success = success
    results['monotonic_steps'] = not flag_1

    # Test 2: Black-box vs white-box (transfer should be WORSE, not better)
    print("\n[Test 2] Black-box transfer vs white-box adaptive attack")
    print("  Expect: white-box >= black-box")
    print("  Red flag: black-box > white-box (gradient is being hidden)")
    # Black-box: craft against model WITHOUT defense
    x_bb = pgd_attack(model, x, y, epsilon, 0.01, 40)
    # White-box: craft WITH defense in the loop
    x_wb = pgd_through_defense(model, defense_fn, x, y, epsilon, 0.01, 40)
    with torch.no_grad():
        success_bb = (model(defense_fn(x_bb)).argmax(1) != y).float().mean().item()
        success_wb = (model(defense_fn(x_wb)).argmax(1) != y).float().mean().item()
    flag_2 = success_bb > success_wb + 0.02
    print(f"  Black-box attack success: {success_bb:.4f}")
    print(f"  White-box attack success: {success_wb:.4f}")
    if flag_2:
        print("  FLAG: black-box > white-box — gradient masking suspected")
    results['whitebox_stronger'] = not flag_2

    # Test 3: Random noise vs FGSM
    print("\n[Test 3] Random noise vs FGSM attack success")
    print("  Expect: FGSM >> random noise")
    print("  Red flag: random noise ≈ FGSM (gradient is uninformative)")
    x_rand = torch.clamp(x + torch.zeros_like(x).uniform_(-epsilon, epsilon), x.min(), x.max())
    x_fgsm_adv = pgd_attack(model, x, y, epsilon, epsilon, 1)  # 1-step = FGSM
    with torch.no_grad():
        success_rand = (model(defense_fn(x_rand)).argmax(1) != y).float().mean().item()
        success_fgsm = (model(defense_fn(x_fgsm_adv)).argmax(1) != y).float().mean().item()
    flag_3 = abs(success_rand - success_fgsm) < 0.05
    print(f"  Random noise attack success: {success_rand:.4f}")
    print(f"  FGSM attack success:         {success_fgsm:.4f}")
    if flag_3:
        print("  FLAG: FGSM ≈ random noise — gradient signal is masked")
    results['fgsm_stronger_than_random'] = not flag_3

    print("\n" + "=" * 55)
    print("Summary:")
    for test, passed in results.items():
        print(f"  {'PASS' if passed else 'FAIL'}: {test}")
    return results


x_b, y_b = next(iter(test_loader))
x_b, y_b = x_b[:128].to(device), y_b[:128].to(device)

results = run_gradient_masking_diagnostics(
    model_clean, gaussian_smooth, x_b, y_b, epsilon=0.3, device=device
)

## 3. TRADES: Separating Clean Accuracy from Robustness

PGD adversarial training (notebook 03) trains entirely on adversarial examples. This trades clean accuracy for robustness — the model can't simultaneously minimize loss on clean examples *and* on worst-case adversarial examples, because those objectives conflict.

TRADES (Zhang et al., 2019) explicitly decomposes the risk:

$$\text{Risk} = \underbrace{\mathbb{E}[\mathcal{L}(f(x), y)]}_\text{natural error} + \underbrace{\frac{1}{\lambda}\mathbb{E}\left[\max_{\delta \in \mathcal{B}} \text{KL}(f(x) \| f(x+\delta))\right]}_\text{boundary error}$$

The boundary error term is minimized when the model's output distribution doesn't change between $x$ and its perturbed versions. This is exactly robustness — the model should be locally consistent.

$\lambda$ controls the tradeoff: small $\lambda$ → more weight on robustness; large $\lambda$ → more weight on clean accuracy. Unlike PGD-AT, TRADES makes this tradeoff explicit and tunable.

In [ ]:
def trades_loss(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    epsilon: float,
    alpha: float,
    n_steps: int,
    beta: float = 6.0,  # 1/lambda — tradeoff: higher beta = more robustness
) -> torch.Tensor:
    """
    TRADES objective: natural loss + beta * KL divergence between clean and adversarial outputs.

    The inner maximization finds the worst-case perturbation in terms of KL divergence,
    not cross-entropy — this is subtly different from PGD-AT.
    """
    model.eval()

    # Inner maximization: find δ that maximizes KL(f(x) || f(x+δ))
    x_adv = x + 0.001 * torch.randn_like(x)  # small random init
    x_adv = x_adv.detach().requires_grad_(True)

    with torch.no_grad():
        p_clean = F.softmax(model(x), dim=1)  # clean output distribution

    for _ in range(n_steps):
        p_adv = F.log_softmax(model(x_adv), dim=1)
        # KL(p_clean || p_adv) — maximize by gradient ascent
        kl = F.kl_div(p_adv, p_clean, reduction='batchmean')
        kl.backward()
        with torch.no_grad():
            x_adv_new = x_adv + alpha * x_adv.grad.sign()
            x_adv_new = torch.clamp(x_adv_new, x - epsilon, x + epsilon)  # L-inf ball around x
            x_adv_new = torch.clamp(x_adv_new, x.min(), x.max())
            x_adv = x_adv_new.requires_grad_(True)

    model.train()
    # Natural loss on clean examples
    loss_natural = F.cross_entropy(model(x), y)
    # Boundary loss: KL divergence between clean and adversarial outputs
    loss_robust = F.kl_div(
        F.log_softmax(model(x_adv.detach()), dim=1),
        F.softmax(model(x), dim=1),
        reduction='batchmean'
    )
    return loss_natural + beta * loss_robust


def train_trades(beta, epochs=5, epsilon=0.3, n_steps=10):
    """Train a model using the TRADES objective."""
    torch.manual_seed(42)
    model = MnistCNN().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    alpha = 2 * epsilon / n_steps

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = trades_loss(model, x, y, epsilon, alpha, n_steps, beta=beta)
            loss.backward()
            opt.step()

    clean_acc = evaluate(model, test_loader, device, n_batches=20)
    # Robust accuracy under PGD-20
    robust_correct = robust_total = 0
    for i, (x, y) in enumerate(test_loader):
        if i >= 10: break
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, epsilon, 2.5 * epsilon / 20, 20)
        with torch.no_grad():
            robust_correct += (model(x_adv).argmax(1) == y).sum().item()
            robust_total += y.size(0)
    robust_acc = robust_correct / robust_total
    return model, clean_acc, robust_acc


print("Training TRADES models at different beta values (beta controls robustness weight)...")
print(f"{'beta':>6} {'Clean acc':>12} {'Robust acc (PGD-20)':>22} {'Gap':>8}")
print("-" * 55)

trades_results = {}
for beta in [1.0, 3.0, 6.0, 10.0]:
    _, clean, robust = train_trades(beta)
    trades_results[beta] = (clean, robust)
    print(f"{beta:>6.1f} {clean:>12.4f} {robust:>22.4f} {clean - robust:>+8.4f}")

## 4. Randomized Smoothing: Certified Robustness

All empirical defenses (adversarial training variants) leave open the possibility that a sufficiently strong attack could defeat them. Randomized smoothing provides a *certificate*: a radius $R$ around each input within which no attack of any kind can change the prediction.

The algorithm:
1. At inference time, add Gaussian noise $\mathcal{N}(0, \sigma^2 I)$ to the input $n$ times
2. Run the base classifier on each noisy input
3. Take the majority vote as the prediction
4. Use a binomial confidence interval to certify that the majority class has probability $\geq p_A$
5. The certificate radius is $R = \frac{\sigma}{2}(\Phi^{-1}(p_A) - \Phi^{-1}(1 - p_A))$

If the $L_2$ perturbation is less than $R$, the smoothed classifier is guaranteed to predict the same class — regardless of the attack.

In [ ]:
class SmoothedClassifier:
    """
    Randomized smoothing wrapper (Cohen et al., ICML 2019).

    Wraps any base classifier and provides:
    - Certifiably robust predictions under L2 perturbations
    - Confidence bounds via binomial statistics
    """
    ABSTAIN = -1  # returned when certification fails

    def __init__(self, base_model: nn.Module, sigma: float, device):
        self.base_model = base_model
        self.sigma = sigma
        self.device = device

    def _sample_predictions(self, x: torch.Tensor, n: int, batch_size: int = 256):
        """Sample n noisy predictions for input x. Returns class count array."""
        self.base_model.eval()
        counts = torch.zeros(10, dtype=torch.long, device=self.device)
        remaining = n
        with torch.no_grad():
            while remaining > 0:
                this_batch = min(remaining, batch_size)
                # Repeat x, add Gaussian noise
                x_batch = x.unsqueeze(0).repeat(this_batch, 1, 1, 1)
                noise = torch.randn_like(x_batch) * self.sigma
                logits = self.base_model(x_batch + noise)
                preds = logits.argmax(dim=1)
                counts += torch.bincount(preds, minlength=10)
                remaining -= this_batch
        return counts

    def certify(self, x: torch.Tensor, n0: int = 100, n: int = 1000, alpha: float = 0.001):
        """
        Predict and certify robustness radius for a single input.

        Args:
            n0:    small sample for prediction (majority vote)
            n:     large sample for certification (confidence bounds)
            alpha: FWER — probability of wrongly certifying

        Returns:
            (prediction, radius) or (ABSTAIN, 0.0) if not certifiable
        """
        from scipy.stats import binom

        # Step 1: small sample to identify top class
        counts0 = self._sample_predictions(x, n0)
        top_class = counts0.argmax().item()

        # Step 2: large sample to bound p_A (probability of top class)
        counts = self._sample_predictions(x, n)
        k_A = counts[top_class].item()  # number of times top class was predicted

        # Lower confidence bound on p_A via binomial test
        p_A_lower = binom.ppf(alpha, n, k_A / n) / n
        p_A_lower = max(k_A / n - scipy_norm.ppf(1 - alpha) * np.sqrt(k_A / n * (1 - k_A / n) / n), 0)

        if p_A_lower < 0.5:
            return self.ABSTAIN, 0.0  # can't certify majority

        # Certificate radius: R = sigma/2 * (Phi^{-1}(p_A) - Phi^{-1}(1-p_A))
        # Simplifies to: R = sigma * Phi^{-1}(p_A)
        radius = self.sigma * scipy_norm.ppf(p_A_lower)
        return top_class, radius

    def predict(self, x: torch.Tensor, n: int = 100):
        """Fast prediction without certification."""
        counts = self._sample_predictions(x, n)
        return counts.argmax().item()


# Train a noise-augmented model (required for randomized smoothing to work well)
SIGMA = 0.5

torch.manual_seed(42)
model_smoothed = MnistCNN().to(device)
opt_s = torch.optim.Adam(model_smoothed.parameters(), lr=1e-3)

print(f"Training Gaussian noise-augmented model (σ={SIGMA}) for smoothing...")
for epoch in range(5):
    model_smoothed.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        x_noisy = x + torch.randn_like(x) * SIGMA  # train with noise
        opt_s.zero_grad()
        F.cross_entropy(model_smoothed(x_noisy), y).backward()
        opt_s.step()

clean_acc = evaluate(model_smoothed, test_loader, device, n_batches=20)
print(f"Base model clean accuracy: {clean_acc:.4f}")

In [ ]:
# Certify robustness for a sample of test examples
smoother = SmoothedClassifier(model_smoothed, sigma=SIGMA, device=device)

test_x, test_y = next(iter(test_loader))
test_x = test_x[:20].to(device)  # certify 20 examples (n=500 per example)
test_y = test_y[:20]

print(f"Certifying 20 examples (σ={SIGMA}, n=500 per example, α=0.001)...")
print(f"{'Idx':>5} {'True':>6} {'Pred':>6} {'Correct':>9} {'Radius':>8} {'Certified':>10}")
print("-" * 50)

radii = []
n_correct = n_certified = n_abstain = 0

for i in range(20):
    pred, radius = smoother.certify(test_x[i], n0=100, n=500, alpha=0.001)
    true = test_y[i].item()
    if pred == smoother.ABSTAIN:
        print(f"{i:>5} {true:>6} {'?':>6} {'—':>9} {'—':>8} {'ABSTAIN':>10}")
        n_abstain += 1
    else:
        correct = (pred == true)
        certified = correct and radius > 0
        print(f"{i:>5} {true:>6} {pred:>6} {'✓' if correct else '✗':>9} {radius:>8.3f} {'✓' if certified else '✗':>10}")
        if correct: n_correct += 1
        if certified: n_certified += 1
        if radius > 0: radii.append(radius)

print()
print(f"Correct:   {n_correct}/20")
print(f"Certified: {n_certified}/20  (correct + positive radius)")
print(f"Abstain:   {n_abstain}/20")
if radii:
    print(f"Mean certified radius: {np.mean(radii):.3f}")
    print(f"  → No L2 attack with ‖δ‖₂ < {np.mean(radii):.3f} can change these predictions")

## 5. The Accuracy-Robustness Tradeoff: Is It Fundamental?

Practitioners often hope that with the right training procedure, you could have both high clean accuracy *and* high robustness. Tsipras et al. (2019) — *"Robustness May Be at Odds with Accuracy"* — showed this hope is likely futile, at least for finite data.

Their argument:
- Standard training learns to exploit *all* features that correlate with the label, including high-frequency, non-robust features that humans don't perceive
- These non-robust features are useful for accuracy (they genuinely correlate with labels) but are adversarially exploitable
- A robust model must *ignore* these features, which costs accuracy
- The gap between clean and robust accuracy is therefore not a training failure — it's a consequence of what the data contains

They provided a simple synthetic dataset where the optimal clean classifier and optimal robust classifier are provably *different* functions. You cannot have both simultaneously.

This reframes adversarial robustness as a *specification problem*: you need to specify which features the model is allowed to use, not just what label it should predict.

In [ ]:
# Visualize the accuracy-robustness tradeoff across methods
# Collect results from all trained models

methods = ['Clean training', 'TRADES β=1', 'TRADES β=3', 'TRADES β=6', 'TRADES β=10', 'Smoothed (σ=0.5)']
clean_accs = [
    evaluate(model_clean, test_loader, device, 20),
    trades_results[1.0][0],
    trades_results[3.0][0],
    trades_results[6.0][0],
    trades_results[10.0][0],
    evaluate(model_smoothed, test_loader, device, 20),
]
robust_accs = [
    None,  # will compute
    trades_results[1.0][1],
    trades_results[3.0][1],
    trades_results[6.0][1],
    trades_results[10.0][1],
    None,  # will compute
]

# Compute robust acc for clean model and smoothed model
def robust_acc_pgd20(model, epsilon=0.3, n_batches=10):
    correct = total = 0
    for i, (x, y) in enumerate(test_loader):
        if i >= n_batches: break
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, epsilon, 2.5 * epsilon / 20, 20)
        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

robust_accs[0] = robust_acc_pgd20(model_clean)
robust_accs[5] = robust_acc_pgd20(model_smoothed)  # PGD doesn't work against smoothing, but shows degradation

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['steelblue', '#2ecc71', '#27ae60', '#1e8449', '#145a32', 'darkorange']
markers = ['o', 's', 's', 's', 's', '^']

for i, (method, clean, robust) in enumerate(zip(methods, clean_accs, robust_accs)):
    ax.scatter(clean, robust, s=150, color=colors[i], marker=markers[i], zorder=3, label=method)
    ax.annotate(method, (clean, robust), textcoords='offset points', xytext=(6, 3), fontsize=8)

# Diagonal: perfect tradeoff line
ax.plot([0.9, 1.0], [0.9, 1.0], 'k--', alpha=0.2, label='Clean = Robust')

ax.set_xlabel('Clean Accuracy')
ax.set_ylabel('Robust Accuracy (PGD-20, ε=0.3)')
ax.set_title('The Accuracy-Robustness Tradeoff')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc='lower left')
ax.set_xlim(0.8, 1.01)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('accuracy_robustness_tradeoff.png', bbox_inches='tight')
plt.show()

print("Moving toward the upper-right corner is the goal — both high clean and robust accuracy.")
print("The gap between the two is the fundamental robustness-accuracy tradeoff.")

## 6. Practical Defense Checklist

When evaluating or implementing a defense against adversarial examples, work through this checklist:

**Before claiming a defense works:**
1. **Evaluate with adaptive attacks** — does the attacker compute gradients through your defense?
2. **Run the gradient masking diagnostics** — do more PGD steps always find stronger attacks?
3. **Report both clean and robust accuracy** — a defense that drops clean accuracy to 50% isn't useful
4. **Specify the threat model** — $L_\infty$? $L_2$? What $\epsilon$? Robustness is always relative to a threat model
5. **Compare to the PGD-AT baseline** — if your defense is more complex than PGD-AT but not more robust, PGD-AT is better

**For production systems:**
- Is your input domain susceptible to adversarial examples? (vision APIs, biometrics: yes; text generation: different concern)
- What threat model is realistic for your deployment? (black-box query access is most common; white-box is worst case)
- Can you use randomized smoothing for a certified guarantee? (costs clean accuracy; requires infrastructure for Monte Carlo)
- At minimum: run the gradient masking diagnostics on any third-party defense before trusting it

In [ ]:
# Defense comparison summary table
import textwrap

defenses = [
    ("Input preprocessing",    "None (provably broken)",  "Low",   "Low",  "Broken by adaptive attacks via BPDA"),
    ("Detection-based",        "None",                    "Low",   "Low",  "Evaded by attacks minimizing detection score"),
    ("FGSM adversarial train", "Empirical",               "Med",   "Med",  "Overfits to FGSM; fails against PGD"),
    ("PGD-AT (Madry)",         "Empirical",               "High",  "High", "Best practical defense; slow training"),
    ("TRADES",                 "Empirical",               "High",  "High", "Better clean/robust tradeoff than PGD-AT"),
    ("Randomized Smoothing",   "Certified (L2)",          "High",  "Med",  "Provable guarantees; slow inference"),
]

print(f"{'Defense':25} {'Guarantee':16} {'Robust':8} {'Cost':6}  Notes")
print("-" * 95)
for name, guarantee, robust, cost, notes in defenses:
    print(f"{name:25} {guarantee:16} {robust:8} {cost:6}  {notes}")

print()
print("Robust = robustness provided | Cost = implementation/training cost")

## Summary

**The three questions for any defense:**
1. Does it survive adaptive attacks? (attacker knows and designs around it)
2. Is the clean accuracy cost acceptable?
3. What threat model does it actually cover?

**What actually works:**
- **PGD adversarial training** — the empirical baseline; everything should beat it to be interesting
- **TRADES** — cleaner objective, better robustness-accuracy tradeoff than vanilla PGD-AT
- **Randomized smoothing** — the only scalable certified defense; real guarantees, real cost

**The fundamental limit:**
The accuracy-robustness tradeoff is provably inherent (Tsipras et al., 2019). Standard training exploits non-robust features because they're useful for accuracy. Robustness requires ignoring them. You cannot fully escape this tradeoff — you can only choose where on the curve to operate.

**Next:** Notebook 05 shifts from the inference surface (adversarial examples) to the privacy surface — membership inference attacks. The question: given only black-box query access to your deployed model, can an attacker determine whether a specific record was in your training set?